# Phase 2 — HyDE + Cross-Encoder Reranker

**Goal**: Close the MRR gap. Phase 1 gave Hit@20=0.74 (matches paper's GPT-4) but MRR@20=0.50 (paper: 0.68).

**Stack added on top of Phase 1**:
1. **HyDE** (Hypothetical Document Embeddings) — LLM generates a fake 'perfect API doc' per query, we embed *that* instead of the query. Fixes query-doc length asymmetry.
2. **BGE-reranker-v2-m3** — cross-encoder reranks top-50 hybrid candidates. Much stronger than bi-encoder for fine-grained ranking.

**Expected result**: Hit@20 ≥ 0.80, MRR@20 ≥ 0.65 — beats paper's GPT-4 on **both** metrics with zero hierarchical LLM calls.

**Pre-reqs**: Phase 1 notebook already ran. Cache files present:
- `/kaggle/working/cache/endpoints.pkl`
- `/kaggle/working/cache/test_queries.json`
- `/kaggle/working/cache/corpus_embs.npy`

**Runtime**: ~15–20 min (warm cache), ~30 min (cold).

## Cell 1 — Installs (only what's new)

Phase 1 already installed most. We re-verify. ~30 sec if cached.

In [ ]:
!pip install -q "httpx==0.27.2" "groq==0.11.0" rank_bm25 sentence-transformers
print("installs verified")

## Cell 2 — Imports, config, resume from Phase 1 cache

In [ ]:
import os, json, random, pickle, time, re
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import defaultdict, Counter

import numpy as np
import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

WORK_DIR = Path("/kaggle/working")
CACHE_DIR = WORK_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True, parents=True)

# Endpoint dataclass MUST be redefined for pickle load
@dataclass
class Endpoint:
    endpoint_id: str
    category: str
    tool_name: str
    tool_description: str
    endpoint_name: str
    endpoint_description: str
    method: str = ""
    parameters: List[Dict] = field(default_factory=list)

    def text_for_embedding(self) -> str:
        parts = [
            f"Category: {self.category}",
            f"API: {self.tool_name}",
            f"API description: {self.tool_description}",
            f"Endpoint: {self.endpoint_name}",
            f"Endpoint description: {self.endpoint_description}",
        ]
        if self.method:
            parts.append(f"Method: {self.method}")
        if self.parameters:
            param_str = ", ".join(p.get("name", "") for p in self.parameters[:10])
            parts.append(f"Parameters: {param_str}")
        return " | ".join(parts)


# Load Phase 1 artifacts
with open(CACHE_DIR / "endpoints.pkl", "rb") as f:
    endpoints: List[Endpoint] = pickle.load(f)
with open(CACHE_DIR / "test_queries.json") as f:
    test_queries = json.load(f)
corpus_embs = np.load(CACHE_DIR / "corpus_embs.npy")

print(f"Loaded {len(endpoints)} endpoints")
print(f"Loaded {len(test_queries)} test queries")
print(f"Loaded corpus embeddings: {corpus_embs.shape}")

## Cell 3 — Groq client (httpx workaround)

In [ ]:
from kaggle_secrets import UserSecretsClient
GROQ_API_KEY = UserSecretsClient().get_secret("GROQ_API_KEY")

import httpx
from groq import Groq

http_client = httpx.Client(timeout=60.0, follow_redirects=True)
client = Groq(api_key=GROQ_API_KEY, http_client=http_client)

resp = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "say ready in 2 words"}],
    max_tokens=10,
)
print("Groq:", resp.choices[0].message.content)

## Cell 4 — Rebuild Phase 1 retrievers (BM25, BGE, hybrid)

BM25 rebuild is fast. BGE embedder reloaded for queries + HyDE embedding.

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9]+", text.lower())

corpus_texts = [e.text_for_embedding() for e in endpoints]

# Try cached BM25, else rebuild
bm25_path = CACHE_DIR / "bm25.pkl"
if bm25_path.exists():
    with open(bm25_path, "rb") as f:
        bm25 = pickle.load(f)
    print("Loaded BM25 from cache")
else:
    print("Building BM25 (2 min)...")
    tokenized_corpus = [tokenize(t) for t in tqdm(corpus_texts, desc="Tokenizing")]
    bm25 = BM25Okapi(tokenized_corpus)
    with open(bm25_path, "wb") as f:
        pickle.dump(bm25, f)

# BGE embedder for queries + HyDE docs
print("Loading BGE-large-en-v1.5...")
embedder = SentenceTransformer("BAAI/bge-large-en-v1.5", device=DEVICE)

# Put corpus embs on GPU for fast matmul
corpus_embs_gpu = torch.from_numpy(corpus_embs).to(DEVICE).float()
print(f"Corpus embs on {DEVICE}: {corpus_embs_gpu.shape}")

def bm25_search(query: str, top_k: int = 50) -> List[Tuple[int, float]]:
    scores = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_idx]

def bge_search(query: str, top_k: int = 50) -> List[Tuple[int, float]]:
    q_emb = embedder.encode(
        [f"Represent this sentence for searching relevant passages: {query}"],
        convert_to_tensor=True,
        normalize_embeddings=True,
    ).to(DEVICE)
    scores = (corpus_embs_gpu @ q_emb.T).squeeze(1)
    top_vals, top_idx = torch.topk(scores, k=top_k)
    return list(zip(top_idx.cpu().tolist(), top_vals.cpu().tolist()))

def rrf_fuse(results_list, k: int = 60, top_k: int = 50):
    scores = defaultdict(float)
    for results in results_list:
        for rank, (doc_id, _) in enumerate(results):
            scores[doc_id] += 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])[:top_k]

def hybrid_search(query: str, top_k: int = 50):
    return rrf_fuse([bm25_search(query, top_k), bge_search(query, top_k)], top_k=top_k)

print("Retrievers ready")

## Cell 5 — HyDE: generate hypothetical docs per query

For each of 50 queries, Groq/Llama-3.3-70B generates a plausible API description. Then we embed the hypothetical doc (not the query). Cached — re-runs instant.

In [ ]:
HYDE_PROMPT = (
    "You are a technical writer documenting a RESTful Web API. "
    "Given the user query below, write a short hypothetical API endpoint "
    "description (40-80 words) that would PERFECTLY answer this query. "
    "Include likely endpoint name, HTTP method, what data it returns, "
    "and typical parameters. Write as if you are documenting a real API. "
    "No preamble, just the description.\n\n"
    "Query: {query}"
)


def generate_hyde(query: str, model="llama-3.3-70b-versatile", retries=3) -> str:
    for i in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": HYDE_PROMPT.format(query=query)}],
                temperature=0.0,
                max_tokens=200,
            )
            return resp.choices[0].message.content.strip()
        except Exception:
            if i == retries - 1:
                return query  # fallback
            time.sleep(2 ** i)
    return query


hyde_cache_path = CACHE_DIR / "hyde_docs.json"
if hyde_cache_path.exists():
    with open(hyde_cache_path) as f:
        hyde_docs = json.load(f)
    print(f"Loaded {len(hyde_docs)} HyDE docs from cache")
else:
    hyde_docs = {}
    for tq in tqdm(test_queries, desc="Generating HyDE docs"):
        q = tq["query"]
        hyde_docs[q] = generate_hyde(q)
    with open(hyde_cache_path, "w") as f:
        json.dump(hyde_docs, f, indent=2)
    print(f"Generated and cached {len(hyde_docs)} HyDE docs")

# Show one example
sample_q = test_queries[0]["query"]
print(f"\nQuery: {sample_q}")
print(f"HyDE doc: {hyde_docs[sample_q]}")

## Cell 6 — Embed HyDE docs + build HyDE retriever

Instead of embedding the query, embed the hypothetical doc. Closer to real doc's embedding space.

In [ ]:
# Embed all HyDE docs
hyde_texts = [hyde_docs[tq["query"]] for tq in test_queries]
hyde_embs = embedder.encode(
    hyde_texts,
    batch_size=32,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).to(DEVICE)
print(f"HyDE embeddings: {hyde_embs.shape}")

# Map query -> hyde embedding
hyde_emb_map = {tq["query"]: hyde_embs[i] for i, tq in enumerate(test_queries)}


def hyde_search(query: str, top_k: int = 50) -> List[Tuple[int, float]]:
    """Dense retrieval using HyDE doc embedding instead of query embedding."""
    if query in hyde_emb_map:
        q_emb = hyde_emb_map[query].unsqueeze(0)
    else:
        # Fallback — generate on the fly
        doc = generate_hyde(query)
        q_emb = embedder.encode([doc], convert_to_tensor=True, normalize_embeddings=True).to(DEVICE)
    scores = (corpus_embs_gpu @ q_emb.T).squeeze(1)
    top_vals, top_idx = torch.topk(scores, k=top_k)
    return list(zip(top_idx.cpu().tolist(), top_vals.cpu().tolist()))


def hyde_hybrid_search(query: str, top_k: int = 50):
    """BM25 + HyDE RRF fusion."""
    return rrf_fuse([bm25_search(query, top_k), hyde_search(query, top_k)], top_k=top_k)


def hyde_bge_hybrid_search(query: str, top_k: int = 50):
    """BM25 + BGE + HyDE tri-fusion."""
    return rrf_fuse([
        bm25_search(query, top_k),
        bge_search(query, top_k),
        hyde_search(query, top_k),
    ], top_k=top_k)

print("HyDE retrievers ready")

## Cell 7 — Load BGE-reranker-v2-m3 (cross-encoder)

568M param XLM-R based reranker. Fits on T4. Downloads ~2.3 GB first time. ~3-5 min first load, then cached.

In [ ]:
from sentence_transformers import CrossEncoder

print("Loading BGE-reranker-v2-m3 (may download ~2.3GB first time)...")
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device=DEVICE)
print("Reranker ready")

# Quick sanity test
pairs = [
    ("weather forecast for a city", "Get current weather conditions for a given city including temperature, humidity, and wind speed."),
    ("weather forecast for a city", "Fetch a list of Pokemon cards with their stats and rarity."),
]
scores = reranker.predict(pairs)
print(f"Relevant pair score: {scores[0]:.4f}")
print(f"Irrelevant pair score: {scores[1]:.4f}")
assert scores[0] > scores[1], "Reranker sanity check failed"

## Cell 8 — Rerank function

Takes any retriever's top-N candidates → reranker scores each (query, doc) pair → returns reranked top-K. This is where most MRR gain comes from.

In [ ]:
def make_reranked_retriever(base_retriever_fn, candidate_k: int = 50):
    """Wrap any retriever with cross-encoder reranking."""
    def retriever(query: str, top_k: int = 20) -> List[Tuple[int, float]]:
        # Step 1: get candidates from base retriever
        candidates = base_retriever_fn(query, top_k=candidate_k)
        if not candidates:
            return []
        # Step 2: build (query, doc) pairs
        cand_ids = [c[0] for c in candidates]
        pairs = [(query, endpoints[cid].text_for_embedding()) for cid in cand_ids]
        # Step 3: score with reranker (batch)
        scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)
        # Step 4: sort by reranker score, return top_k
        reranked = sorted(zip(cand_ids, scores), key=lambda x: -x[1])[:top_k]
        return [(cid, float(s)) for cid, s in reranked]
    return retriever


# Build reranked variants
hybrid_reranked = make_reranked_retriever(hybrid_search, candidate_k=50)
hyde_hybrid_reranked = make_reranked_retriever(hyde_hybrid_search, candidate_k=50)
triple_reranked = make_reranked_retriever(hyde_bge_hybrid_search, candidate_k=50)

# Sanity check on one query
test_q = test_queries[0]["query"]
out = triple_reranked(test_q, top_k=5)
print(f"Query: {test_q}")
for cid, sc in out:
    print(f"  [{sc:.3f}] {endpoints[cid].endpoint_id} — {endpoints[cid].endpoint_description[:80]}")

## Cell 9 — Evaluation harness (same as Phase 1)

In [ ]:
def evaluate(retriever_fn, test_queries, endpoints, ks=(5, 10, 20)) -> Dict[str, float]:
    id_to_idx = {e.endpoint_id: i for i, e in enumerate(endpoints)}
    max_k = max(ks)
    hits = {k: 0 for k in ks}
    mrrs = {k: 0.0 for k in ks}
    n = len(test_queries)

    for tq in tqdm(test_queries, desc="Evaluating", leave=False):
        gold_idx = id_to_idx.get(tq["gold_endpoint_id"])
        if gold_idx is None:
            continue
        results = retriever_fn(tq["query"], top_k=max_k)
        ranked_ids = [r[0] for r in results]
        rank = None
        for r, doc_id in enumerate(ranked_ids, start=1):
            if doc_id == gold_idx:
                rank = r
                break
        for k in ks:
            if rank is not None and rank <= k:
                hits[k] += 1
                mrrs[k] += 1.0 / rank

    return {f"Hit@{k}": hits[k] / n for k in ks} | {f"MRR@{k}": mrrs[k] / n for k in ks}

print("Eval harness ready")

## Cell 10 — Run Phase 2 ablation

Runs 5 variants; reranker variants take ~1-2 min each (cross-encoder cost). Total ~8-10 min.

In [ ]:
all_results: Dict[str, Dict[str, float]] = {}

print("[1/5] Hybrid (Phase 1 baseline)")
all_results["hybrid"] = evaluate(hybrid_search, test_queries, endpoints)
print("   ", all_results["hybrid"])

print("\n[2/5] HyDE + BM25 (no BGE)")
all_results["hyde_hybrid"] = evaluate(hyde_hybrid_search, test_queries, endpoints)
print("   ", all_results["hyde_hybrid"])

print("\n[3/5] HyDE + BGE + BM25 (triple fusion)")
all_results["triple_fusion"] = evaluate(hyde_bge_hybrid_search, test_queries, endpoints)
print("   ", all_results["triple_fusion"])

print("\n[4/5] Hybrid + Reranker")
all_results["hybrid_rerank"] = evaluate(hybrid_reranked, test_queries, endpoints)
print("   ", all_results["hybrid_rerank"])

print("\n[5/5] Triple Fusion + Reranker (Phase 2 full)")
all_results["triple_fusion_rerank"] = evaluate(triple_reranked, test_queries, endpoints)
print("   ", all_results["triple_fusion_rerank"])

# Save
with open(CACHE_DIR / "phase2_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

## Cell 11 — Final comparison table (Paper + Phase 1 + Phase 2)

In [ ]:
paper_sbert = {"Hit@5": 0.48, "MRR@5": 0.3503, "Hit@10": 0.56, "MRR@10": 0.3609,
               "Hit@20": 0.64, "MRR@20": 0.3668}
paper_gpt4  = {"Hit@5": 0.60, "MRR@5": 0.5073, "Hit@10": 0.66, "MRR@10": 0.5673,
               "Hit@20": 0.74, "MRR@20": 0.6773}

# Phase 1 results (recomputed)
phase1_hybrid = all_results["hybrid"]

rows = [
    ("Paper SBERT", paper_sbert),
    ("Paper GPT-4 (target)", paper_gpt4),
    ("P1: Hybrid (BM25+BGE)", phase1_hybrid),
    ("P2: + HyDE hybrid", all_results["hyde_hybrid"]),
    ("P2: Triple fusion", all_results["triple_fusion"]),
    ("P2: Hybrid + Rerank", all_results["hybrid_rerank"]),
    ("P2: Triple + Rerank (FULL)", all_results["triple_fusion_rerank"]),
]

print("\n===== FULL COMPARISON =====")
header = f"{'Method':<30} {'Hit@5':<8} {'MRR@5':<8} {'Hit@10':<8} {'MRR@10':<8} {'Hit@20':<8} {'MRR@20':<8}"
print(header)
print("-" * len(header))
for name, r in rows:
    print(f"{name:<30} "
          f"{r['Hit@5']:<8.4f} {r['MRR@5']:<8.4f} "
          f"{r['Hit@10']:<8.4f} {r['MRR@10']:<8.4f} "
          f"{r['Hit@20']:<8.4f} {r['MRR@20']:<8.4f}")

# Delta vs paper's best
best = all_results["triple_fusion_rerank"]
print("\n===== DELTA vs Paper's GPT-4 best =====")
for k in ["Hit@5", "MRR@5", "Hit@10", "MRR@10", "Hit@20", "MRR@20"]:
    delta = best[k] - paper_gpt4[k]
    sign = "+" if delta >= 0 else ""
    print(f"  {k}: ours={best[k]:.4f} vs paper={paper_gpt4[k]:.4f}  ({sign}{delta:.4f})")

print("\nPhase 2 complete. Paste this output back for Phase 3 planning.")